In [24]:
# ===============================
# 1. Imports
# ===============================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np



In [25]:
# ===============================
# 2. Load and clean data
# ===============================

dataset_path = "/Users/catherineli/Downloads/dataset.csv"
df_read = pd.read_csv(dataset_path)

# drop missing values
df_read = df_read.dropna()

# drop exact duplicate rows
df_read = df_read.drop_duplicates()

# drop unnamed column
df = df_read.drop(columns=["Unnamed: 0"])

# remove tempo = 0
df = df[df["tempo"] != 0]

# remove duplicate songs by same artist, keeping most popular version
df = (
    df.sort_values("popularity", ascending=False)
      .drop_duplicates(subset=["track_name", "artists"], keep="first")
)

# remove songs with popularity = 0
df = df[df["popularity"] > 0]

# Count songs in each genre
genre_counts = df["track_genre"].value_counts()

# Keep only genres with at least 400 songs
valid_genres = genre_counts[genre_counts >= 400].index

# Filter dataframe
df = df[df["track_genre"].isin(valid_genres)]

# Reset index
df = df.reset_index(drop=True)

print("Number of genres left:", df["track_genre"].nunique())
print("Number of songs left:", len(df))

df.head()


Number of genres left: 93
Number of songs left: 70937


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,2tTmW7RDtMQtBk7m2rYeSw,Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52","Quevedo: Bzrp Music Sessions, Vol. 52",99,198937,False,0.621,0.782,2,-5.548,1,0.0440,0.01250,0.033000,0.2300,0.550,128.033,4,hip-hop
1,4uUG5RXrOk84mYEfFvj3cK,David Guetta;Bebe Rexha,I'm Good (Blue),I'm Good (Blue),98,175238,True,0.561,0.965,7,-3.673,0,0.0343,0.00383,0.000007,0.3710,0.304,128.040,4,pop
2,4LRPiXqCikLlN15c3yImP7,Harry Styles,As It Was,As It Was,95,167303,False,0.520,0.731,6,-5.338,0,0.0557,0.34200,0.001010,0.3110,0.662,173.930,4,pop
3,6xGruZOHLs39ZbVccQTuPZ,Joji,Glimpse of Us,Glimpse of Us,94,233456,False,0.440,0.317,8,-9.258,1,0.0531,0.89100,0.000005,0.1410,0.268,169.914,3,pop
4,3JvKfv6T31zO0ini8iNItO,Tom Odell,Long Way Down (Deluxe),Another Love,93,244360,True,0.445,0.537,4,-8.532,0,0.0400,0.69500,0.000017,0.0944,0.131,122.769,4,pop


In [26]:
# ===============================
# 3. Choose content-based features
# ===============================


features = [
     "danceability","loudness",
    "speechiness", "acousticness",
    "instrumentalness", "liveness",
    "valence", "tempo","popularity","key", "mode",
]


In [ ]:
# ===============================
# 2. Prepare data
# ===============================

X = df[features].values
y = df["track_genre"].values # target variable for classification

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) # standardize features

label_encoder = LabelEncoder()
y= label_encoder.fit_transform(y) # encode genres as integers

# PyTorch CNN shape: samples, channels, features
X = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1]) # add channel dimension for CNN

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2) # split into train/test sets

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long) # convert NumPy arrays into PyTorch tensors

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True) # create DataLoader for training
test_loader = DataLoader(TensorDataset(X_test, y_test),batch_size=64) # create DataLoader for testing

In [ ]:
# ===============================
# 3. CNN model
# ===============================

class SongCNN(nn.Module):
    def __init__(self, num_genres):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=2),
            nn.ReLU()
        ) # create feature patterns

        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 3, 64),
            nn.ReLU()
        ) # turns the CNN output into a 64-dimensional embedding

        self.classifier = nn.Linear(64, num_genres) # final layer for genre classification

    def forward(self, x):
        x = self.conv(x)
        emb = self.embedding(x)
        out = self.classifier(emb)
        return out  # return both the genre prediction and the embedding

    def get_embedding(self, x):
        x = self.conv(x)
        emb = self.embedding(x)
        return emb # method to get the embedding for a given input


num_genres = len(label_encoder.classes_) # get number of unique genres for output layer
model = SongCNN(num_genres) # instantiate the model

In [43]:
# ===============================
# 4. Train model
# ===============================
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    total_loss = 0

    for batch_X, batch_y in dataloader:
        # prediction
        pred = model(batch_X)

        # loss
        loss = loss_fn(pred, batch_y)

        # backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Train Loss: {total_loss:.4f}")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    test_loss = 0
    correct = 0
    size = len(dataloader.dataset)

    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            pred = model(batch_X)
            test_loss += loss_fn(pred, batch_y).item()
            correct += (pred.argmax(1) == batch_y).type(torch.float).sum().item()

    test_loss /= len(dataloader)
    accuracy = correct / size

    print(f"Test Accuracy: {accuracy:.4f}, Test Loss: {test_loss:.4f}")



In [39]:
# ===============================
# 5. Test accuracy and loss function
# ===============================

loss_fn = nn.CrossEntropyLoss()
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

epochs = 10

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, model, loss_fn, optimizer)
    test_loop(test_loader, model, loss_fn)



Epoch 1
-------------------------------
Train Loss: 2173.9773
Test Accuracy: 0.3361, Test Loss: 2.5197
Epoch 2
-------------------------------
Train Loss: 2164.0666
Test Accuracy: 0.3356, Test Loss: 2.5174
Epoch 3
-------------------------------
Train Loss: 2158.4109
Test Accuracy: 0.3393, Test Loss: 2.5106
Epoch 4
-------------------------------
Train Loss: 2153.8218
Test Accuracy: 0.3366, Test Loss: 2.5165
Epoch 5
-------------------------------
Train Loss: 2147.4217
Test Accuracy: 0.3383, Test Loss: 2.5150
Epoch 6
-------------------------------
Train Loss: 2143.6075
Test Accuracy: 0.3392, Test Loss: 2.5129
Epoch 7
-------------------------------
Train Loss: 2141.0175
Test Accuracy: 0.3405, Test Loss: 2.5101
Epoch 8
-------------------------------
Train Loss: 2134.5572
Test Accuracy: 0.3361, Test Loss: 2.5105
Epoch 9
-------------------------------
Train Loss: 2132.9463
Test Accuracy: 0.3382, Test Loss: 2.5215
Epoch 10
-------------------------------
Train Loss: 2126.9944
Test Accur

In [ ]:
# ===============================
# 6. Get song embeddings
# ===============================

X_all = torch.tensor(X, dtype=torch.float32) # convert entire dataset to tensor for embedding extraction

model.eval() # set model to evaluation mode

with torch.no_grad():
    song_embeddings = model.get_embedding(X_all).numpy() # get 64-dimensional embeddings for all songs in the dataset

In [41]:
# ===============================
# 5. KNN Classifier
# ===============================

knn_cnn = NearestNeighbors(n_neighbors=11,metric="cosine") # instantiate KNN with cosine similarity to find similar songs based on embeddings

knn_cnn.fit(song_embeddings) 


,n_neighbors,11
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [ ]:
def cnn_knn_recommend(song_name, n_recommendations=5):
    song_name = song_name.lower() # convert input song name to lowercase for case-insensitive matching

    matches = df[df["track_name"].str.lower().str.contains(song_name, na=False)] # find songs in the dataframe that match the input name (case-insensitive)

    if matches.empty:
        return "Song not found."

    idx = matches.index[0] # get index of the first matching song in the dataframe

    target_embedding = song_embeddings[idx].reshape(1, -1) # get the embedding for the target song and reshape for KNN input

    distances, indices = knn_cnn.kneighbors(
        target_embedding,
        n_neighbors=n_recommendations + 1
    ) # get nearest neighbors (including the song itself)

    recommendations = []

    for dist, i in zip(distances[0], indices[0]):
        if i == idx:
            continue # skip the input song itself

        recommendations.append({
            "track_name": df.iloc[i]["track_name"],
            "artists": df.iloc[i]["artists"],
            "genre": df.iloc[i]["track_genre"],
            "similarity": 1 - dist
        }) # convert cosine distance to similarity score

    return pd.DataFrame(recommendations) 

cnn_knn_recommend("Come Back To Me", 5)

,track_name,artists,genre,similarity
0,Homeworld,Enei;Charli Brix,drum-and-bass,0.986942
1,Homie Don't Play That,Lenzman,drum-and-bass,0.979640
2,This Is La,Lemon D,drum-and-bass,0.978403
3,Anye Up,Ed Solo,breakbeat,0.974311
4,Move On,Matrix & Futurebound;Cat Knight,drum-and-bass,0.971022
